### Weniger Bias, mehr Rendite? Hybrides Modell aus Large Language Models und Machine Learning zur Analyse von Earnings Calls
**Bachelorarbeit an der Universität St.Gallen (HSG)**\
[© 2026 Ruben Cardell](https://polyformproject.org/licenses/noncommercial/1.0.0)

---

# Preprocessing – Versuche 1 und 2

Dieses Notebook bereitet die Rohdaten der Earnings Calls für die Verarbeitung
durch das LLM auf:

1. Rohdatensatz laden und inspizieren
2. Fehlende Unternehmensnamen ergänzen (Voraussetzung für die Anonymisierung)
3. Frage-Antwort-Paare aus den Transkripten extrahieren
4. Unternehmensnamen per Fuzzy-Matching anonymisieren
5. Datensätze für Versuch 1 (einzelne Frage-Antwort-Paare) und Versuch 2
   (gesamter Frage-Antwort-Teil je Transkript) exportieren
6. Token-Anzahl der LLM-Läufe berechnen

**Versionsbezeichnungen im Code:** `v3` = Versuch 1, `v4` = Versuch 2,
`v5` = Versuch 3 (Kombination beider Datensätze).

In [ ]:
# Google Drive einbinden (Ausführung in Google Colab)
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


## 1 Datensatz laden und inspizieren

In [ ]:
# Rohdatensatz: Earnings-Call-Transkripte des S&P 500 (2005–2025)
df = pd.read_parquet('/content/drive/MyDrive/Bachelorarbeit/data/raw_hf.parquet')

In [ ]:
# Spalten, Datentypen und fehlende Werte prüfen
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33362 entries, 0 to 33361
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   symbol              33362 non-null  object 
 1   company_name        24377 non-null  object 
 2   company_id          24377 non-null  float64
 3   year                33362 non-null  int64  
 4   quarter             33362 non-null  int64  
 5   date                33362 non-null  object 
 6   structured_content  33362 non-null  object 
 7   earnings_date       33362 non-null  object 
 8   episode_id          33362 non-null  object 
dtypes: float64(1), int64(2), object(6)
memory usage: 2.3+ MB


In [ ]:
# Erste Zeilen zur Sichtprüfung
df.head()

,symbol,company_name,company_id,year,quarter,date,structured_content,earnings_date,episode_id
0,A,"Agilent Technologies, Inc.",154924.0,2006,1,2006-02-13 14:45:00,[{'speaker': 'Agilent Technologies Incorporate...,2006-02-13,A_2006_Q1
1,A,"Agilent Technologies, Inc.",154924.0,2006,2,2006-05-15 16:30:00,"[{'speaker': 'Executives', 'text': 'Hilliard C...",2006-05-15,A_2006_Q2
2,A,"Agilent Technologies, Inc.",154924.0,2006,3,2006-08-14 16:30:00,"[{'speaker': 'Executives', 'text': 'Hilliard C...",2006-08-14,A_2006_Q3
3,A,"Agilent Technologies, Inc.",154924.0,2006,4,2006-11-14 08:00:00,"[{'speaker': 'Executives', 'text': 'Hilliard C...",2006-11-14,A_2006_Q4
4,A,"Agilent Technologies, Inc.",154924.0,2007,1,2007-02-15 16:30:00,"[{'speaker': 'TRANSCRIPT SPONSOR', 'text': ''}...",2007-02-15,A_2007_Q1


In [ ]:
# Beispieltranskript: strukturierter Inhalt als Liste von Sprecher-Turns
print(df[df["episode_id"] == "NKE_2011_Q4"]["structured_content"].iloc[0])

[{'speaker': 'Executives', 'text': 'Eric Sprunk - Vice President of Merchandising and Product Gary DeStefano - President of Global Operations Donald Blair - Chief Financial Officer and Vice President Jeanne Jackson - President of Direct to Consumer Kelley Hall - Senior Director of Investor Relations Trevor Edwards - Vice President of Global Brand and Category Management Roger Wyett - President of Nike Affiliates Leading Business and Strategic Direction Unknown Executive -  Charlie Denson - President of Nike Brand Mark Parker - Chief Executive Officer, President, Executive Director and Member of Executive Committee'}
 {'speaker': 'Analysts', 'text': 'Brian McGough - Morgan Stanley Eric Tracy - FBR Capital Markets & Co. Maggie Gilliam - Gilliam and Company Paul Swinand - Morningstar Inc. Taposh Bari - Jefferies & Company, Inc. Kate McShane - Citigroup Inc Faye Landes - Consumer Edge Research, LLC Sam Poser - Sterne Agee & Leach Inc. Mitchel Kummetz - Robert W. Baird & Co. Incorporated Om

## 2 Fehlende Unternehmensnamen ergänzen

Die Anonymisierung setzt für jedes Transkript den offiziellen Firmennamen
voraus. Fehlende Namen werden zuerst automatisch über yfinance abgerufen und
anschliessend manuell ergänzt.

In [ ]:
# yfinance liefert zu einem Ticker den hinterlegten Unternehmensnamen
import yfinance as yf

In [ ]:
# Fehlende Unternehmensnamen automatisch über yfinance abrufen
missing_mask = df["company_name"].isna()
unique_symbols = df.loc[missing_mask, "symbol"].unique()
print(f"Tickers missing: {len(unique_symbols)}")
print(unique_symbols)

# Je Ticker nur eine Abfrage, Ergebnisse zwischenspeichern
name_cache = {}
for symbol in unique_symbols:
    try:
        name = yf.Ticker(symbol).info.get("shortName")
        if name:
            name_cache[symbol] = name
            print(f"{symbol} -> {name}")
        else:
            print(f"Could not fetch name for {symbol}")
    except Exception as e:
        # Delistete oder umbenannte Ticker sind über yfinance nicht mehr auflösbar
        print(f"Could not fetch name for {symbol}: {e}")

df.loc[missing_mask, "company_name"] = df.loc[missing_mask, "symbol"].map(name_cache)

Tickers missing: 114
['AA' 'AAL' 'AFL' 'ALTR' 'AN' 'ANF' 'ASH' 'AT' 'BBBY' 'CAR' 'CHIR' 'CL'
 'CMCSA' 'COOP' 'COP' 'COST' 'CSCO' 'CSX' 'CTAS' 'CVS' 'D' 'DIS' 'DNB'
 'DRI' 'DVA' 'EA' 'EBAY' 'EFX' 'EL' 'ES' 'ETN' 'ETR' 'ETSY' 'EXC' 'F'
 'FDX' 'FE' 'FMCC' 'FRC' 'GE' 'GEN' 'GIS' 'GLW' 'GM' 'GME' 'GOOGL' 'GS'
 'GWW' 'HAL' 'HFC' 'HPQ' 'HRB' 'HST' 'IAC' 'JBL' 'JNJ' 'JNPR' 'JPM' 'JWN'
 'K' 'KBH' 'KO' 'KODK' 'KR' 'KSU' 'LUV' 'M' 'MAR' 'MAT' 'MBI' 'MCD' 'MDLZ'
 'MNK' 'MO' 'MSFT' 'MU' 'NEM' 'NKE' 'NTAP' 'NVDA' 'ODP' 'ORCL' 'PARA'
 'PEG' 'PEP' 'PG' 'PLD' 'POM' 'PPG' 'QCOM' 'RF' 'RL' 'SBUX' 'STT' 'SUN'
 'SWK' 'TGT' 'TIF' 'TPR' 'USB' 'VIAV' 'VLO' 'VRSN' 'VZ' 'WBA' 'WCG' 'WELL'
 'WEN' 'WFC' 'WMB' 'XEL' 'XLNX' 'XOM' 'YUM']
AA -> Alcoa Corporation
AAL -> American Airlines Group, Inc.
AFL -> AFLAC Incorporated
Could not fetch name for ALTR
AN -> AutoNation, Inc.
ANF -> Abercrombie & Fitch Company


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AT"}}}


ASH -> Ashland Inc.
Could not fetch name for AT
BBBY -> Bed Bath & Beyond, Inc.
CAR -> Avis Budget Group, Inc.
CHIR -> Global X Funds Global X MSCI Ch
CL -> Colgate-Palmolive Company
CMCSA -> Comcast Corporation
Could not fetch name for COOP
COP -> ConocoPhillips
COST -> Costco Wholesale Corporation
CSCO -> Cisco Systems, Inc.
CSX -> CSX Corporation
CTAS -> Cintas Corporation
CVS -> CVS Health Corporation
D -> Dominion Energy, Inc.
DIS -> Walt Disney Company (The)


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DNB"}}}


Could not fetch name for DNB
DRI -> Darden Restaurants, Inc.
DVA -> DaVita Inc.
EA -> Electronic Arts Inc.
EBAY -> eBay Inc.
EFX -> Equifax, Inc.
EL -> Estee Lauder Companies, Inc. (T
ES -> Eversource Energy (D/B/A)
ETN -> Eaton Corporation, PLC
ETR -> Entergy Corporation
ETSY -> Etsy, Inc.
EXC -> Exelon Corporation
F -> Ford Motor Company
FDX -> FedEx Corporation
FE -> FirstEnergy Corp.
FMCC -> Freddie Mac
Could not fetch name for FRC
GE -> GE Aerospace
GEN -> Gen Digital Inc.
GIS -> General Mills, Inc.
GLW -> Corning Incorporated
GM -> General Motors Company
GME -> GameStop Corporation
GOOGL -> Alphabet Inc.
GS -> Goldman Sachs Group, Inc. (The)
GWW -> W.W. Grainger, Inc.
HAL -> Halliburton Company


...

Could not fetch name for ODP


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PARA"}}}


ORCL -> Oracle Corporation
Could not fetch name for PARA
PEG -> Public Service Enterprise Group
PEP -> Pepsico, Inc.
PG -> Procter & Gamble Company (The)
PLD -> Prologis, Inc.
POM -> POMDOCTOR LIMITED
PPG -> PPG Industries, Inc.
QCOM -> QUALCOMM Incorporated
RF -> Regions Financial Corporation
RL -> Ralph Lauren Corporation
SBUX -> Starbucks Corporation
STT -> State Street Corporation
SUN -> Sunoco LP
SWK -> Stanley Black & Decker, Inc.
TGT -> Target Corporation
Could not fetch name for TIF
TPR -> Tapestry, Inc.
USB -> U.S. Bancorp
VIAV -> Viavi Solutions Inc.
VLO -> Valero Energy Corporation
VRSN -> VeriSign, Inc.


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


VZ -> Verizon Communications Inc.
Could not fetch name for WBA
Could not fetch name for WCG
WELL -> Welltower Inc.
WEN -> Wendy's Company (The)
WFC -> Wells Fargo & Company
WMB -> Williams Companies, Inc. (The)
XEL -> Xcel Energy Inc.
Could not fetch name for XLNX
XOM -> ExxonMobil Holdings Corporation
YUM -> Yum! Brands, Inc.


In [ ]:
# Verbleibende Lücken nach dem automatischen Abruf prüfen
print(f'Still missing: {df[df["company_name"].isna()]["symbol"].unique()}')

In [ ]:
# Manuell recherchierte Namen für die von yfinance nicht auflösbaren Ticker
# (überwiegend Unternehmen, die im Beobachtungszeitraum delistet oder
# übernommen wurden)
company_names = {
    "ABMD": "Abiomed, Inc.",
    "ADS": "Alliance Data Systems",
    "AKS": "AK Steel Holding Corporation",
    "ALTR": "Altera Corp.",
    "ALXN": "Alexion Pharmaceuticals",
    "AMBC": "Ambac Financial Group Inc.",
    "AT": "Alltel Corp.",
    "ATGE": "Adtalem Global Education",
    "ATVI": "Activision Blizzard",
    "BIG": "Big Lots",
    "CBH": "Commerce Bancorp",
    "CERN": "Cerner Corporation",
    "CIT": "CIT Group Inc.",
    "CMA": "Comerica Incorporated",
    "COOP": "Mr. Cooper Group",
    "CSX": "CSX Corporation",
    "CTLT": "Catalent, Inc.",
    "CTRA": "Coterra Energy",
    "CXO": "Concho Resources Inc.",
    "DFS": "Discover Financial Services",
    "DISCK": "Discovery, Inc.",
    "DISH": "DISH Network",
    "DNB": "Dun & Bradstreet",
    "DO": "Diamond Offshore Drilling",
    "DRE": "Duke Realty Corporation",
    "ENDP": "Endo International plc",
    "ETFC": "E*TRADE Financial Corporation",
    "FI": "Fiserv, Inc.",
    "FL": "Foot Locker",
    "FLIR": "FLIR Systems",
    "FRC": "First Republic Bank",
    "HBI": "Hanesbrands Inc.",
    "HFC": "HollyFrontier Corporation",
    "IGT": "International Game Technology",
    "IPG": "The Interpublic Group of Companies, Inc.",
    "JNPR": "Juniper Networks",
    "JWN": "Nordstrom, Inc.",
    "K": "Kellogg Company",
    "KSU": "Kansas City Southern",
    "LM": "Legg Mason",
    "LSI": "LSI Corporation",
    "MDP": "Meredith Corporation",
    "MMC": "Marsh McLennan",
    "MNK": "Mallinckrodt Pharmaceuticals",
    "MRO": "Marathon Oil Corporation",
    "MTW": "The Manitowoc Company, Inc.",
    "MXIM": "Maxim Integrated Products",
    "NBL": "Noble Energy, Inc.",
    "NLSN": "Nielsen Holdings",
    "ODP": "The ODP Corporation",
    "PARA": "Paramount Global",
    "PBCT": "People's United Financial",
    "PDCO": "Patterson Companies, Inc.",
    "PLL": "Pall Corporation",
    "PXD": "Pioneer Natural Resources",
    "QEP": "QEP Resources",
    "RRD": "R.R. Donnelley & Sons Company",
    "SEE": "Sealed Air Corporation",
    "SIVB": "SVB Financial Group",
    "SNV": "Synovus Financial Corp.",
    "SRCL": "Stericycle Inc.",
    "SWK": "Stanley Black & Decker, Inc.",
    "TGNA": "TEGNA Inc.",
    "TIF": "Tiffany & Co.",
    "TRV": "The Travelers Companies, Inc.",
    "TSS": "Total System Services",
    "TWTR": "Twitter, Inc.",
    "VAR": "Varian Medical Systems",
    "VIAB": "Viacom Inc.",
    "WBA": "Walgreens Boots Alliance",
    "WCG": "WellCare Health Plans, Inc.",
    "WPX": "WPX Energy",
    "WRK": "WestRock Company",
    "WYND": "Wyndham Destinations, Inc.",
    "X": "United States Steel Corporation",
    "XEC": "Cimarex Energy Co.",
    "XLNX": "Xilinx",
}

In [ ]:
# Manuell recherchierte Namen in die verbleibenden Lücken einsetzen
df["company_name"] = df["company_name"].fillna(df["symbol"].map(company_names))

In [ ]:
# Stichprobe: Zuordnung für einen manuell ergänzten Ticker kontrollieren
df[df["symbol"] == "AT"]

,symbol,company_name,company_id,year,quarter,date,structured_content,earnings_date,episode_id
3238,AT,Alltel Corp.,NaN,2005,4,2006-01-22 17:00:00,"[{'speaker': 'Operator', 'text': 'Good morning...",2006-01-22,AT_2005_Q4
3239,AT,Alltel Corp.,NaN,2006,1,2006-05-03 17:00:00,"[{'speaker': 'Operator', 'text': 'Good morning...",2006-05-03,AT_2006_Q1
3240,AT,Alltel Corp.,NaN,2006,2,2006-08-03 17:00:00,"[{'speaker': 'Operator', 'text': 'At this time...",2006-08-03,AT_2006_Q2
3241,AT,Alltel Corp.,NaN,2006,3,2006-10-27 17:00:00,"[{'speaker': 'Operator', 'text': 'Good morning...",2006-10-27,AT_2006_Q3
3242,AT,Alltel Corp.,NaN,2006,4,2007-02-20 17:00:00,"[{'speaker': 'Operator', 'text': 'Good morning...",2007-02-20,AT_2006_Q4
3243,AT,Alltel Corp.,NaN,2007,1,2007-04-27 17:00:00,"[{'speaker': 'Operator', 'text': 'Good morning...",2007-04-27,AT_2007_Q1
3244,AT,Alltel Corp.,NaN,2007,2,2007-08-02 17:00:00,"[{'speaker': 'Operator', 'text': 'Good morning...",2007-08-02,AT_2007_Q2
3245,AT,Alltel Corp.,NaN,2007,3,2007-10-22 17:00:00,"[{'speaker': 'Operator', 'text': 'I would like...",2007-10-22,AT_2007_Q3


## 3 Frage-Antwort-Splitting

In [ ]:
def extract_qa_pairs(structured_content):
    """Extrahiert die Frage-Antwort-Paare aus einem Earnings-Call-Transkript.

    Der Operator kündigt in Earnings Calls jeden Analysten an und ist in den
    Transkripten einheitlich benannt. Seine Turns dienen daher als Trennmarken
    zwischen den Frage-Antwort-Paaren. Die vorbereitete Einleitung des
    Managements wird verworfen, da sie andere sprachliche Signale enthält als
    die spontanen Antworten im Frage-Antwort-Teil.

    Rückgabe: Liste von Segmenten, je Segment eine Liste von Sprecher-Turns.
    Leere Liste, falls kein Operator ausgewiesen ist (741 Transkripte).
    """
    try:
        turns = structured_content.tolist()
        assert all("speaker" in t and "text" in t for t in turns)
    except Exception:
        return []

    # Metadaten-Turns entfernen, die nur die Teilnehmerliste aufzählen
    IGNORED_SPEAKERS = {"executives", "analysts", "presentation"}
    turns = [t for t in turns if str(t["speaker"]).strip().lower() not in IGNORED_SPEAKERS]

    # Segmente an den Operator-Turns trennen; der Operator selbst wird verworfen
    segments, current = [], []
    for t in turns:
        if str(t["speaker"]).strip().lower() == "operator":
            segments.append(current)
            current = []

            # Ausnahme: Enthält der Operator-Turn eine Frage, gehört er zum Segment
            if "?" in str(t["text"]):
              current.append(t)
        else:
            current.append(t)
    segments.append(current)

    # Das erste nicht-leere Segment ist immer die Einleitung des Managements und
    # wird übersprungen, auch wenn es zufällig eine (rhetorische) Frage enthält.
    first_nonempty = next((i for i, seg in enumerate(segments) if seg), None)
    if first_nonempty is None:
        return []

    # Nur Segmente mit mindestens einer Frage und einer Antwort behalten
    return [seg for seg in segments[first_nonempty + 1:] if len(seg) > 1]

## 4 Anonymisierung

In [ ]:
# RapidFuzz stellt das Token-Set-Verhältnis für das Fuzzy-Matching bereit
!pip install rapidfuzz

In [ ]:
import re
import numpy as np
from rapidfuzz import fuzz
import string

In [ ]:
# Begriffe, die zwar in Firmennamen vorkommen, diese aber nicht eindeutig
# identifizieren. Sie werden vor dem Fuzzy-Matching aus dem Firmennamen
# entfernt und nie einzeln ersetzt, damit sie im Fliesstext unangetastet
# bleiben. Die Liste wurde iterativ aus der Häufigkeitsauswertung der
# protokollierten Ersetzungen aufgebaut (Appendix A).
GENERIC_WORDS = {
    # Stopp- und Funktionswörter
    "the", "of", "and", "a", "an", "i", "us", "on", "all", "over", "one",
    "first", "second", "third", "fourth", "fifth", "sixth",
    "in", "as", "at", "for", "by", "to", "is", "was", "are", "be", "or",
    "that", "this", "these", "those", "its", "our", "new", "old",
    "general", "half", "range", "best", "home", "year", "total",
    "income", "public", "big", "live", "super", "state", "states",

    # Rechtsformen und Firmenzusätze
    "inc", "incorporated", "llc", "ltd", "limited", "plc",
    "corp", "corporation", "corporations", "co", "company", "companies",
    "group", "groups", "holding", "holdings", "enterprises", "trust", "bros",

    # generische Branchen- und Geschäftsbegriffe
    "systems", "solutions", "technologies", "technology",
    "industries", "industry", "international", "national", "global", "worldwide",
    "financial", "capital", "services", "service", "brands", "brand",
    "business", "businesses", "energy", "health", "healthcare",
    "hotels", "hotel", "news", "oil", "gas", "water", "life", "steel",
    "foods", "food", "beauty", "pool", "pools", "paper", "glass",
    "controls", "automation", "aerospace", "drilling", "insurance",
    "real", "estate", "apartment", "apartments", "communities", "community",
    "restaurants", "restaurant", "cruise", "cruises", "airlines", "airline",
    "medical", "surgical", "diagnostics", "biotech", "pizza",
    "beverage", "beverages", "freight", "centers", "center", "exchange",
    "markets", "market", "management", "investment", "investments", "equity",
    "design", "digital", "data", "analytics", "research", "network", "networks",
    "power", "materials", "material", "chemical", "chemicals",
    "tire", "tires", "tower", "towers", "property", "properties",
    "storage", "waste", "products", "product", "parts", "part",
    "stores", "store", "works", "resources", "realty",

    # Banken und Finanzinstitute
    "bank", "banks", "bancorp", "bancorporation", "bancshares", "banc", "buy",
    "savings", "mortgage", "mortgages", "credit", "lending", "fund", "funds",

    # weitere generische Begriffe aus der Häufigkeitsauswertung
    "air", "line", "lines", "express", "host", "street", "quest",
    "dynamics", "motor", "motors", "mobile", "platform", "platforms",
    "supply", "rental", "rentals", "residential", "payment", "payments",
    "booking", "bookings", "sciences", "science", "laboratory", "laboratories",
    "pharmaceutical", "pharmaceuticals", "petroleum", "resort", "resorts",
    "leasing", "lease", "leases", "transport", "transportation", "logistics",
    "distribution", "wholesale", "retail", "apparel", "tobacco", "spirits",

    # mehrdeutige Begriffe (Herkunft, Richtung, Währung)
    "american", "united", "southern", "western", "northern", "eastern",
    "royal", "dollar", "west", "east", "north", "south", "republic", "union",
    "enterprise", "america",

    # häufige Satzanfänge und Füllwörter
    "they", "them", "then", "due", "being", "really", "looking",
    "corporate", "margin", "morning",
}

In [ ]:
def anonymize_transcript(transcript, company_name, replacement_word="Blahblahblah", threshold=80):
    """Ersetzt alle Nennungen des Unternehmensnamens in einem Transkript.

    Verfahren nach Glasserman und Lin (2023), erweitert für Fliesstext: Der
    Firmenname wird um Satzzeichen, Rechtsformen und generische Begriffe
    bereinigt. Anschliessend wird über alle Wortfolgen des Transkripts iteriert
    und per Fuzzy-Matching (Token-Set-Verhältnis) mit dem bereinigten Namen
    verglichen. Ab einer Ähnlichkeit über dem Schwellenwert gilt die Wortfolge
    als Treffer und wird durch den Platzhalter ersetzt. Die Wortfolgen werden
    von der längsten zur kürzesten geprüft, damit eine vollständige Nennung als
    Einheit ersetzt und nicht in ihre Teilwörter zerlegt wird. Zusätzlich wird
    bei mehrteiligen Namen nach den Initialen als Abkürzung gesucht (z.B. "IBM").

    Rückgabe: (anonymisiertes Transkript, Anzahl Ersetzungen, ersetzte Zeichenfolgen).
    """
    cleaned_name = re.sub(r'[^\w\s]', '', company_name)
    name_words = [w for w in cleaned_name.split() if w.lower() not in GENERIC_WORDS]
    n_words = len(name_words)
    abbr = ''.join(w[0] for w in name_words) if n_words >= 2 else None

    # Besteht ein Name ausschliesslich aus generischen Begriffen (z.B. "Western
    # Union"), darf nicht auf die einzelnen Wörter zurückgefallen werden: sonst
    # würden "Company" oder "Southern" im gesamten Fliesstext ersetzt. Stattdessen
    # wird nur noch die vollständige Wortfolge als Einheit gesucht.
    fallback_full_phrase_only = False
    if n_words == 0:
        name_words = [w for w in cleaned_name.split() if w]
        n_words = len(name_words)
        if n_words == 0:
            return np.array(transcript), 0, []
        fallback_full_phrase_only = True
        abbr = ''.join(w[0] for w in name_words) if n_words >= 2 else None

    replacement_count = 0
    replaced_strings = []
    new_transcript = []

    for entry in transcript:
        words = entry['text'].split()

        # Von der längsten Wortfolge bis hinunter zu einem einzelnen Wort
        lengths_to_try = [n_words] if fallback_full_phrase_only else range(n_words, 0, -1)

        for length in lengths_to_try:
            name_candidates = [' '.join(name_words[j:j+length]) for j in range(n_words - length + 1)]
            i = 0
            while i <= len(words) - length:
                substring = ' '.join(words[i:i+length])
                stripped = substring.strip(string.punctuation)

                # Schutz 1: rein generische Wörter nie einzeln ersetzen
                if stripped.lower() in GENERIC_WORDS:
                    i += 1
                    continue

                # Schutz 2: nur grossgeschriebene Wortfolgen prüfen, da Firmennamen
                # Eigennamen sind; verhindert Treffer bei gewöhnlichen Wörtern
                if not stripped[:1].isupper():
                    i += 1
                    continue

                # Beste Übereinstimmung über alle Teilfolgen des Firmennamens
                best_score = max(fuzz.token_set_ratio(cand.lower(), substring.lower()) for cand in name_candidates)
                if best_score > threshold:
                    if substring not in replaced_strings:
                        replaced_strings.append(substring)
                    words[i:i+length] = [replacement_word]
                    replacement_count += 1
                i += 1

        text = ' '.join(words)

        # Abkürzung aus den Initialen als eigenständiges Wort ersetzen
        if abbr:
            matches = re.findall(r'\b' + re.escape(abbr) + r'\b', text)
            replaced_strings.extend(matches)
            text = re.sub(r'\b' + re.escape(abbr) + r'\b', replacement_word, text)
            replacement_count += len(matches)

        new_transcript.append({**entry, 'text': text})

    return np.array(new_transcript), replacement_count, replaced_strings

### 4.1 Fine-Tuning der Anonymisierung

Die Liste der generischen Begriffe wurde iterativ aufgebaut: Nach jedem
Durchlauf der Pipeline (Abschnitt 5) werden die protokollierten Ersetzungen
nach Häufigkeit ausgewertet und jene Begriffe in `GENERIC_WORDS` ergänzt, die
zwar Bestandteil eines Firmennamens sind, im Fliesstext aber ein gewöhnliches
Branchen-, Rechtsform- oder Funktionswort darstellen.

In [ ]:
import ast
from collections import Counter

In [ ]:
# Alle protokollierten Ersetzungen aller Transkripte in eine Liste zusammenführen
# (bei aus CSV geladenen Daten stehen die Listen als String und müssen zuerst
# über ast.literal_eval zurückgewandelt werden)
#all_words = [word for row in stats_df["anonym_replaced"]
#             for word in ast.literal_eval(row)]
all_words = [word for row in stats_df["anonym_replaced"] for word in row]

# Häufigkeiten zählen
word_counts = Counter(all_words)

In [ ]:
# Die 1'000 häufigsten Treffer manuell durchsehen und Fehltreffer als
# generische Begriffe in GENERIC_WORDS ergänzen
top_words = word_counts.most_common(1000)
top_words

[('Starbucks', 4235),
 ('FedEx', 3234),
 ('Ford', 2969),
 ('Adobe', 2904),
 ('Allstate', 2635),
 ('Visa', 2614),
 ('BD', 2602),
 ('NIKE', 2522),
 ('Apple', 2252),
 ("Moody's", 2051),
 ('Monster', 2043),
 ('Nucor', 2027),
 ('GE', 1988),
 ('NextEra', 1912),
 ('Ameren', 1896),
 ('Entergy', 1893),
 ('EA', 1886),
 ('BlackRock', 1826),
 ('Equifax', 1766),
 ('Dow', 1756),
 ('Aflac', 1701),
 ('Principal', 1661),
 ('Noble', 1654),
 ('Aon', 1648),
 ('Walmart', 1624),
 ('Oracle', 1617),
 ('PACCAR', 1595),
 ('Garden', 1556),
 ('Loews', 1497),
 ('Wynn', 1492),
 ('Akamai', 1470),
 ('NRG', 1456),
 ('Cisco', 1450),
 ('Disney', 1355),
 ('Southwest', 1353),
 ('Snap-on', 1349),
 ('Etsy', 1342),
 ('NetApp', 1342),
 ('Dominion', 1341),
 ('Unum', 1340),
 ('CME', 1307),
 ('Discovery', 1290),
 ('Nike', 1290),
 ('Microsoft', 1289),
 ('ANSYS', 1287),
 ('TI', 1276),
 ('Sysco', 1265),
 ('Broadridge', 1259),
 ('NVIDIA', 1229),
 ('Salesforce', 1223),
 ('CSX', 1222),
 ('Johnson &', 1216),
 ('Cognizant', 1213),
 ('EO

## 5 Pipeline für den gesamten Datensatz

Anonymisierung und Frage-Antwort-Splitting werden für jedes Transkript
nacheinander ausgeführt. Ergebnis sind zwei Tabellen: die Statistiken je
Transkript und der Datensatz für Versuch 1 mit einer Zeile je
Frage-Antwort-Paar.

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
stats_rows = []
new_rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    # Zuerst anonymisieren, dann in Frage-Antwort-Paare aufteilen
    transcript, count, replaced = anonymize_transcript(row["structured_content"], row["company_name"])
    qa_pairs = extract_qa_pairs(transcript)

    # Jedes Segment als Fliesstext "Sprecher: Text" für den User-Prompt formatieren
    prompts = ['\n'.join(f'{turn["speaker"]}: {turn["text"]}' for turn in seg) for seg in qa_pairs]


    # Kennzahlen je Transkript zur Kontrolle der Anonymisierung protokollieren
    stats_rows.append({
        "episode_id": row["episode_id"],
        "company_name": row["company_name"],
        "anonym_counts": count,
        "anonym_replaced": replaced,
        "qa_pairs_count": len(qa_pairs),
    })

    # Eine Zeile je Frage-Antwort-Paar; sequence nummeriert die Paare im Call
    if len(qa_pairs) != 0:
        for seq, qa_pair in enumerate(prompts, start=1):
            new_rows.append({
                "episode_id": row["episode_id"],
                "sequence": seq,
                "qa_pair": qa_pair,
            })

stats_df = pd.DataFrame(stats_rows)
new_df = pd.DataFrame(new_rows)

  0%|          | 0/33362 [00:00<?, ?it/s]

## 6 Pipeline-Statistiken auswerten

In [ ]:
# Erste Zeilen der Statistiken zur Sichtprüfung
stats_df.head(10)

,episode_id,company_name,anonym_counts,anonym_replaced,qa_pairs_count
0,A_2006_Q1,"Agilent Technologies, Inc.",32,"[Agilent, Agilent’s, Agilent,, Agilent., Agile...",8
1,A_2006_Q2,"Agilent Technologies, Inc.",31,"[Agilent, Agilent’s, Agilent,, Agilent., Agile...",7
2,A_2006_Q3,"Agilent Technologies, Inc.",35,"[Agilent, Agilent’s, Agilent., Agilent,]",9
3,A_2006_Q4,"Agilent Technologies, Inc.",22,"[Agilent, Agilent’s, Agilent., Agilent,]",7
4,A_2007_Q1,"Agilent Technologies, Inc.",15,"[Agilent, Agilent's, Agilent,, Agilent.]",11
5,A_2007_Q2,"Agilent Technologies, Inc.",16,"[Agilent, Agilent's, Agilent,, Agilent.]",10
6,A_2007_Q3,"Agilent Technologies, Inc.",18,"[Agilent, Agilent's, Agilent’s, Agilent,]",8
7,A_2007_Q4,"Agilent Technologies, Inc.",21,"[Agilent, Agilent's, Agilent., Agilent,]",9
8,A_2008_Q1,"Agilent Technologies, Inc.",29,"[Agilent, Agilent's, Agilent,, Agilents', Agil...",10
9,A_2008_Q2,"Agilent Technologies, Inc.",28,"[Agilent, Agilent's, Agilent,, Agilent’s]",9


In [ ]:
# Verteilung der Ersetzungen und der Anzahl Frage-Antwort-Paare je Transkript
stats_df.describe()

,anonym_counts,qa_pairs_count
count,33362.000000,33362.000000
mean,15.205473,10.230562
std,14.563919,3.869861
min,0.000000,0.000000
25%,5.000000,8.000000
50%,12.000000,10.000000
75%,21.000000,13.000000
max,238.000000,37.000000


In [ ]:
# Extremfälle prüfen: Unternehmen, deren Name zugleich ein Produktname ist,
# werden besonders häufig genannt
stats_df[stats_df["anonym_counts"] >= 200]

,episode_id,company_name,anonym_counts,anonym_replaced,qa_pairs_count
22227,NKE_2011_Q4,"NIKE, Inc.",206,"[Nike, NIKE, Nike+, Nike., Nike,, NIKE,, NIKE....",0
23688,PARA_2020_Q4,Paramount Global,238,"[Paramount, Paramount+,, Paramount+., Paramoun...",1
28964,TGT_2024_Q4,Target Corporation,206,"[Target's, Target, Target., Target,, Target:, ...",1


In [ ]:
# Stichprobe aus dem Datensatz für Versuch 1
new_df[112360:112370]

,episode_id,sequence,qa_pair
112360,EL_2012_Q2,7,Constance Marie Maneaty - BMO Capital Markets ...
112361,EL_2012_Q2,8,"Linda Bolton-Weiser - Caris & Company, Inc., R..."
112362,EL_2012_Q2,9,"Javier Escalante - Consumer Edge Research, LLC..."
112363,EL_2012_Q2,10,"John A. Faucher - JP Morgan Chase & Co, Resear..."
112364,EL_2012_Q2,11,"William Schmitz - Deutsche Bank AG, Research D..."
112365,EL_2012_Q3,1,"Christopher Ferrara - BofA Merrill Lynch, Rese..."
112366,EL_2012_Q3,2,"Neely J.N. Tamminga - Piper Jaffray Companies,..."
112367,EL_2012_Q3,3,"Mark S. Astrachan - Stifel, Nicolaus & Co., In..."
112368,EL_2012_Q3,4,David Wu - Telsey Advisory Group LLC: On Europ...
112369,EL_2012_Q3,5,Alice Beebe Longley - The Buckingham Research ...


In [ ]:
# Einzelnes Frage-Antwort-Paar im Volltext prüfen
print(new_df.iloc[212360]["qa_pair"])

Scott Davis: Good morning, everybody.
Mike Roman: Hi, Scott.
Scott Davis: The price increase in the quarter was a pretty dramatic quarter-to-quarter sequential change, but we've clearly seen pretty big step-ups in CPI and PPI. What -- when you think about 4Q and kind of think about the ramp into 2022, do you continue with this aggressive of a ramp. Is there any mitigating factors that holds you back from getting price? I mean, I don't think I remember you guys being as far below CPI before. But these are strange times too. So I'm not sure it really matters. Just some color on that ramp would be helpful. Thank you.
Monish Patolawala: Sure, Scott. As I mentioned in my prepared remarks, you will see us continue to gain price in the fourth quarter. I know they've been quite a few questions around why it has taken us this long to get price. As we mentioned before, we follow up pretty methodical approach of taking consistent, taking price across geographies. We work with our customers. In ce

## 7 Speichern und Laden

In [ ]:
# Statistiken der Anonymisierung speichern
stats_df.to_csv('/content/drive/MyDrive/Bachelorarbeit/data/prep_stats_v3.csv', index=False)

In [ ]:
# Datensatz für Versuch 1 speichern (eine Zeile je Frage-Antwort-Paar)
new_df.to_csv('/content/drive/MyDrive/Bachelorarbeit/data/prep_data_v3.csv', index=False)

In [ ]:
# Statistiken laden
stats_df = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/prep_stats_v3.csv')

In [ ]:
# Datensatz für Versuch 1 laden
new_df = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/prep_data_v3.csv')

## 8 Datensatz für Versuch 2 vorbereiten

Für Versuch 2 werden die Frage-Antwort-Paare eines Transkripts wieder zu einem
zusammenhängenden Text verbunden, sodass das LLM den gesamten
Frage-Antwort-Teil in einem Aufruf bewertet.

In [ ]:
# Alle Paare eines Transkripts zu einem Text zusammenfügen
df_v4 = new_df.groupby('episode_id')['qa_pair'].apply('\n\n'.join).reset_index()

In [ ]:
# Anzahl Transkripte und fehlende Werte prüfen
df_v4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32621 entries, 0 to 32620
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   episode_id  32621 non-null  object
 1   qa_pair     32621 non-null  object
dtypes: object(2)
memory usage: 509.8+ KB


In [ ]:
# Erste Zeilen zur Sichtprüfung
df_v4.head()

,episode_id,qa_pair
0,AAL_2007_Q3,"Frank Boroch - BearStearns: Tom, if you could ..."
1,AAL_2007_Q4,Gary Chase – Lehman Brothers: If you don’t hav...
2,AAL_2008_Q1,Michael Linenberg – Merrill Lynch: A couple of...
3,AAL_2008_Q2,Daniel Cravens: With us in the room today is D...
4,AAL_2008_Q3,"Kevin Christy – UBS: Hey, thank you very much...."


In [ ]:
# Vollständigen Frage-Antwort-Teil eines Transkripts prüfen
print(df_v4.iloc[0]["qa_pair"])

Frank Boroch - BearStearns: Tom, if you could maybe give us a sense of how unit revenueperformed across the hubs in the quarter, that would be helpful.
Tom Horton: Our domestic unit revenue in the quarter was up 4.8%. As youbreak that down, it was up about 5.5% to TransCON, Dallaswas up about 2.8%, Chicago was up3.8%, Miami was up a little over 3%, and St. Louis was up almost 8%.
Frank Boroch - BearStearns: You alluded to the policy change, the expiration policychange coming in December. Can you give us a sense of the magnitude, therevenue impact you're expecting from the 18-month expiration period?
Tom Horton: We're still working on it. It will depend on the results ofthat work, but it will be measured in the tens of millions of dollars inrevenue.
Frank Boroch - BearStearns: Is there anything happening in the Atlantic division, interms of, your revenue seems to be underperforming a little bit the broaderindustry?
Tom Horton: I think that's a fair question. Our Trans-Atlanticperformanc

In [ ]:
# Datensatz für Versuch 2 speichern (eine Zeile je Transkript)
df_v4.to_csv('/content/drive/MyDrive/Bachelorarbeit/data/prep_data_v4.csv', index=False)

## 9 Token-Anzahl berechnen

Die Token-Anzahl dient der Abschätzung von Laufzeit und Kosten der LLM-Läufe
sowie der Festlegung von `MAX_MODEL_LEN` in `main_v3.py` und `main_v4.py`.

In [ ]:
from transformers import AutoTokenizer

In [ ]:
# System-Prompt für Versuch 1 (Appendix B): einzelne Frage-Antwort-Paare
instructions_v3 = """You are a helpful assistant. Your job is to analyze the sentiment, tone, and emotion of an earnings call question-answer pair. Measure each dimension on a scale of 0 to 1 in increments of 0.1.
- Relevant
0 = The answer does not address the question — the speaker deflects, changes subject, or responds with generic/unrelated information.
1 = The answer directly addresses the specific question asked, staying on-topic and matching its scope.

- Clear
0 = The answer is vague, ambiguous, or contradictory — a listener could not confidently restate the speaker's main point.
1 = The answer is direct and unambiguous — a listener could confidently restate the speaker's main point in one sentence.

- Optimistic
0 = The speaker frames future outcomes negatively or neutrally — emphasizing headwinds, declines, challenges, or "wait and see" language.
1 = The speaker frames future outcomes positively — emphasizing growth, improvement, opportunity, or favorable expectations.

- Specific
0 = The answer relies on generic, qualitative language with no concrete figures, dates, or technical detail (e.g., "strong performance," "on track," "good progress").
1 = The answer includes at least one concrete number, percentage, date, metric, or technical/operational detail supporting the claim.

- Cautious
0 = The speaker takes a bold or committed stance — no acknowledgment of risk, downside scenarios, or uncertainty; commitments are unqualified.
1 = The speaker explicitly acknowledges risk, downside scenarios, or uncertainty, or hedges a commitment/guidance (e.g., "depending on," "if conditions permit," "we can't guarantee").

- Assertive
0 = The speaker uses uncertain or hedged language about what will happen (e.g., "might," "could," "we hope," "possibly").
1 = The speaker uses definitive, confident language about what will happen (e.g., "will," "is," "we are confident that").

- Valence
0 = The content is negative — words signaling dissatisfaction, disappointment, frustration, or distress.
1 = The content is positive — words signaling satisfaction, happiness, pride, or contentment.

- Arousal
0 = The tone is calm and low-energy — measured, flat, unhurried delivery, regardless of whether the content is positive or negative.
1 = The tone is high-energy and intense — excited, urgent, or agitated delivery, regardless of whether the content is positive or negative.

- Dominance
0 = The speaker frames the company as controlled by external forces — outcomes depend on the market, regulators, customers, or circumstances beyond their control.
1 = The speaker frames the company as in control — actively driving outcomes through their own decisions and actions.

Additionally, predict the performance of the companies stock price over the following 90 days. Give an estimate in percent like this: 0.05 (= 5% up); -0.05 (= 5% down)

Output the values in json format like this. Do not output any explanation or other content, just the json as text.
{"relevant": value, "clear": value, "optimistic": value, "specific": value, "cautious": value, "assertive": value, "valence": value, "arousal": value, "dominance": value, "price": value}
"""

In [ ]:
# System-Prompt für Versuch 2 (Appendix C): gesamter Frage-Antwort-Teil,
# auf mehrere Sprecher bezogen formuliert
instructions_v4 = """You are a helpful assistant. Your job is to analyze the sentiment, tone, and emotion of an earnings call transcript. Measure each dimension on a scale of 0 to 1 in increments of 0.1.
- Relevant
0 = The answers do not address the questions — the speaker deflects, changes subject, or responds with generic/unrelated information.
1 = The answers directly address the specific questions asked, staying on-topic and match its scope.

- Clear
0 = The answers are vague, ambiguous, or contradictory — a listener could not confidently restate the speaker's main point.
1 = The answers are direct and unambiguous — a listener could confidently restate the speaker's main point in one sentence.

- Optimistic
0 = The speakers frame future outcomes negatively or neutrally — emphasizing headwinds, declines, challenges, or "wait and see" language.
1 = The speakers frame future outcomes positively — emphasizing growth, improvement, opportunity, or favorable expectations.

- Specific
0 = The answers rely on generic, qualitative language with no concrete figures, dates, or technical detail (e.g., "strong performance," "on track," "good progress").
1 = The answers include concrete numbers, percentages, dates, metrics, or technical/operational detail supporting the claims.

- Cautious
0 = The speakers take a bold or committed stance — no acknowledgment of risk, downside scenarios, or uncertainty; commitments are unqualified.
1 = The speakers explicitly acknowledge risk, downside scenarios, or uncertainty, or hedge a commitment/guidance (e.g., "depending on," "if conditions permit," "we can't guarantee").

- Assertive
0 = The speakers use uncertain or hedged language about what will happen (e.g., "might," "could," "we hope," "possibly").
1 = The speakers use definitive, confident language about what will happen (e.g., "will," "is," "we are confident that").

- Valence
0 = The content is negative — words signaling dissatisfaction, disappointment, frustration, or distress.
1 = The content is positive — words signaling satisfaction, happiness, pride, or contentment.

- Arousal
0 = The tone is calm and low-energy — measured, flat, unhurried delivery, regardless of whether the content is positive or negative.
1 = The tone is high-energy and intense — excited, urgent, or agitated delivery, regardless of whether the content is positive or negative.

- Dominance
0 = The speakers frame the company as controlled by external forces — outcomes depend on the market, regulators, customers, or circumstances beyond their control.
1 = The speakers frame the company as in control — actively driving outcomes through their own decisions and actions.

Additionally, predict the performance of the companies stock price over the following 1, 30, and 90 days. Give an estimate in percent like this: 0.05 (= 5% up); -0.05 (= 5% down)

Output the values in json format like this. Do not output any explanation or other content, just the json as text.
{"relevant": value, "clear": value, "optimistic": value, "specific": value, "cautious": value, "assertive": value, "valence": value, "arousal": value, "dominance": value, "price_1d": value, "price_30d": value, "price_90d": value}
"""

In [ ]:
# Ausgaben des bereits gelaufenen Versuchs 1, um die Output-Tokens zu zählen
output_df = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/output_v3.csv')

In [ ]:
# Token-Anzahl Versuch 1

MODEL_NAME = "google/gemma-4-E4B-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Input-Tokens je Frage-Antwort-Paar
input_lengths = []
for text in tqdm(new_df["qa_pair"].fillna("").astype(str), desc="Tokenizing input"):
    input_lengths.append(len(tokenizer.encode(text)))

input = pd.Series(input_lengths)

# Output-Tokens je erzeugtem JSON-Objekt
output_lengths = []
for text in tqdm(output_df["output"].astype(str), desc="Tokenizing output"):
    output_lengths.append(len(tokenizer.encode(text)))

output = pd.Series(output_lengths)

# Der System-Prompt fällt bei jedem Aufruf erneut an
system_tok = len(tokenizer.encode(instructions_v3))

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

Tokenizing input:   0%|          | 0/341312 [00:00<?, ?it/s]

Tokenizing output:   0%|          | 0/341312 [00:00<?, ?it/s]

In [ ]:
# Gesamtverbrauch Versuch 1: Texte plus System-Prompt je Aufruf
count = len(input)
input_tok = input.sum()
input_total = input_tok + (count * system_tok)
output_tok = output.sum()

print(f"Count:  {count:,}")
print(f"Transcripts:  {input_tok:,}")
print(f"System Prompt:  {system_tok:,}")
print("---")
print(f"Total Input:  {input_total:,}")
print(f"Total Output:  {output_tok:,}")

Count:  341,312
Transcripts:  232,663,881
System Prompt:  721
---
Total Input:  478,749,833
Total Output:  29,870,895


In [ ]:
# Token-Anzahl Versuch 2

MODEL_NAME = "google/gemma-4-E4B-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Input-Tokens je gesamtem Frage-Antwort-Teil
input_lengths = []
for text in tqdm(df_v4["qa_pair"].fillna("").astype(str), desc="Tokenizing input"):
    input_lengths.append(len(tokenizer.encode(text)))

input = pd.Series(input_lengths)


system_tok = len(tokenizer.encode(instructions_v4))

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

Tokenizing input:   0%|          | 0/32621 [00:00<?, ?it/s]

In [ ]:
# Gesamtverbrauch Versuch 2; das Maximum bestimmt MAX_MODEL_LEN in main_v4.py
count = len(input)
input_tok = input.sum()
input_total = input_tok + (count * system_tok)

print(f"Count:  {count:,}")
print(f"Transcripts:  {input_tok:,}")
print(f"System Prompt:  {system_tok:,}")
print("---")
print(f"Max Input: {input.max():,}")
print(f"Total Input:  {input_total:,}")

# Die Output-Tokens sind proportional zur Datensatzgrösse -> rund 10% von Versuch 1

Count:  32,621
Transcripts:  232,972,572
System Prompt:  737
---
Max Input: 27,079
Total Input:  257,014,249
